# Recommendation System

In [1]:
# Importing Required Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import  MinMaxScaler
from scipy.sparse import hstack
from sklearn.metrics.pairwise import cosine_similarity

## Data Preprocessing

### Load the Dataset

In [2]:
df = pd.read_csv("anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


### Column Description:
- **anime_id**: Unique identifier for each anime.
- **name**: Title of the anime.
- **genre**: Genre(s) associated with the anime (multiple genres per anime).
- **type**: Broadcast type such as TV, Movie, OVA, etc.
- **episodes**: Number of episodes in the anime.
- **rating**: Average user rating of the anime.
- **members**: Number of community members who have interacted with the anime.

### Dataset Shape

In [3]:
df.shape

(12294, 7)

### Data Types & Statistics

In [4]:
df.info()
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  str    
 2   genre     12232 non-null  str    
 3   type      12269 non-null  str    
 4   episodes  12294 non-null  str    
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 1.3 MB


,anime_id,name,genre,type,episodes,rating,members
count,12294.000000,12294,12232,12269,12294,12064.000000,1.229400e+04
unique,NaN,12292,3264,6,187,NaN,NaN
top,NaN,Saru Kani Gassen,Hentai,TV,1,NaN,NaN
freq,NaN,2,823,3787,5677,NaN,NaN
mean,14058.221653,NaN,NaN,NaN,NaN,6.473902,1.807134e+04
std,11455.294701,NaN,NaN,NaN,NaN,1.026746,5.482068e+04
min,1.000000,NaN,NaN,NaN,NaN,1.670000,5.000000e+00
25%,3484.250000,NaN,NaN,NaN,NaN,5.880000,2.250000e+02
50%,10260.500000,NaN,NaN,NaN,NaN,6.570000,1.550000e+03
75%,24794.500000,NaN,NaN,NaN,NaN,7.180000,9.437000e+03


### Handle Missing Values

In [5]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [6]:
df['genre'] = df['genre'].fillna('Unknown_Genre')
df['type'] = df['type'].fillna('Other')
df['rating'] = df['rating'].fillna(df['rating'].median())

###  Handle "Unknown" values

In [7]:
unknown_count_per_column = (df == "Unknown").sum()
unknown_count_per_column

anime_id      0
name          0
genre         0
type          0
episodes    340
rating        0
members       0
dtype: int64

In [8]:
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [9]:
df['genre_list'] = df['genre'].apply(
    lambda x: [g.strip() for g in x.split(',')] if isinstance(x, str) else []
)
df[['genre', 'genre_list']].head()

,genre,genre_list
0,"Drama, Romance, School, Supernatural","[Drama, Romance, School, Supernatural]"
1,"Action, Adventure, Drama, Fantasy, Magic, Mili...","[Action, Adventure, Drama, Fantasy, Magic, Mil..."
2,"Action, Comedy, Historical, Parody, Samurai, S...","[Action, Comedy, Historical, Parody, Samurai, ..."
3,"Sci-Fi, Thriller","[Sci-Fi, Thriller]"
4,"Action, Comedy, Historical, Parody, Samurai, S...","[Action, Comedy, Historical, Parody, Samurai, ..."


- **Genre & Type:** Missing categorical values were filled with `"Unknown"` to retain all
  anime records and avoid dropping potentially useful items from the recommendation pool.
- **Episodes:** Missing or non-numeric episode values were replaced with the median number
  of episodes to reduce the impact of extreme values.
- **Rating:** Missing ratings were imputed using the median rating to ensure robustness
  against skewed distributions and outliers.

This approach ensures minimal data loss while maintaining statistical consistency.

## Feature Selection and TF-IDF Encoding

To compute similarity between anime, both categorical and numerical features were selected
to capture content, popularity, and engagement characteristics.

### Selected Features:
- **Genre**: Represents the thematic content of an anime. Since an anime can belong to
  multiple genres, this feature is encoded using **TF-IDF vectorization**, which converts
  genre text into weighted numerical vectors.
- **Rating**: Acts as a popularity and quality signal, reflecting overall user preference.
  It is treated as a numerical feature and normalized.
- **Episodes**: Represents the scale and length of the anime, distinguishing short and
  long-running series. This numerical feature is normalized.
- **Members**: Indicates community engagement and popularity. Higher values suggest
  greater audience interest and are normalized to avoid dominance.

### Encoding and Normalization:
- **TF-IDF vectorization** is applied to the genre feature to assign weighted importance
  to genres and reduce the influence of very common genres.
- Numerical features (rating, episodes, members) are scaled using **Min–Max normalization**
  to ensure comparable value ranges.
- The encoded and normalized features are combined to form the final feature matrix used
  for cosine similarity computation.

### TF-IDF vectorization

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
genre_tfidf = tfidf.fit_transform(df['genre'])

### Scale Numerical Features (Min-Max Scaling)

In [11]:
num_features = df[['rating', 'members', 'episodes']]
scaler = MinMaxScaler()
num_scaled = scaler.fit_transform(num_features)

### Combining All Features

In [12]:
final_features = hstack([genre_tfidf, num_scaled])

## Recommendation System (Cosine Similarity)

### Compute Cosine Similarity

In [13]:
cosine_sim = cosine_similarity(final_features)

###  Create Anime Index Mapping

In [14]:
anime_index = pd.Series(df.index, index=df["name"]).drop_duplicates()

### Recommendation Function

In [15]:
def recommend_anime(title, top_n=10, threshold=0.05):
    """
    Recommends similar anime using cosine similarity.

    Parameters:
    title (str): Anime name for which recommendations are required
    top_n (int): Number of recommendations to return
    threshold (float): Minimum cosine similarity score

    Returns:
    DataFrame or str: Recommended anime list or error message
    """

    # Create a case-insensitive index mapping
    anime_index = {name.lower(): idx for idx, name in enumerate(df['name'])}
    title = title.lower()

    # Validate input anime
    if title not in anime_index:
        return "Anime not found in dataset."

    # Get index of the target anime
    idx = anime_index[title]

    # Compute similarity scores excluding the anime itself
    sim_scores = [
        (i, score)
        for i, score in enumerate(cosine_sim[idx])
        if i != idx and score >= threshold
    ]

    # Sort by similarity score (descending) and select top N
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:top_n]

    # Handle case where no recommendations satisfy threshold
    if len(sim_scores) == 0:
        return "No recommendations found. Try lowering the threshold."

    # Extract indices and similarity scores
    anime_indices = [i for i, _ in sim_scores]
    scores = [score for _, score in sim_scores]

    # Prepare final result DataFrame
    result = df.loc[
        anime_indices, ['name', 'genre', 'rating', 'type']
    ].copy()

    result['similarity_score'] = scores

    # Reset index for clean presentation
    return result.reset_index(drop=True)

## Performance Analysis of the Recommendation System

The recommendation system is based on item–item cosine similarity, where anime are
recommended according to their content similarity and popularity-related features.

The system performs well in identifying anime with similar genres, as genre information
encoded using TF-IDF plays a primary role in the similarity computation. This ensures that
the recommended anime are thematically relevant to the target anime.

Numerical features such as rating, number of members, and episode count further enhance
recommendation quality by prioritizing higher-rated, more popular, and richer-content anime
among similar items.

Since the system is purely content-based, it does not incorporate individual user
preferences. As a result, the same set of recommendations is generated for all users given
a particular anime.

### Areas of Improvement:
- Incorporating user–anime interaction data to develop a hybrid recommendation system.
- Personalizing recommendations using user feedback or watch history.
- Applying dimensionality reduction techniques to improve scalability for larger datasets.

### Generate Recommendations for a Target Anime

In [16]:
recommend_anime("Naruto", threshold=0.05)

,name,genre,rating,type,similarity_score
0,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94,TV,0.991495
1,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32,TV,0.942790
2,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",8.16,TV,0.915894
3,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53,Movie,0.905891
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50,Movie,0.905552
5,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03,Movie,0.901962
6,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58,OVA,0.884480
7,Naruto Soyokazeden Movie: Naruto to Mashin to ...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.11,Movie,0.884093
8,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.95,TV,0.882997
9,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68,Special,0.882098


In [17]:
idx = df[df['name'] == "Naruto"].index[0]
sorted(cosine_sim[idx], reverse=True)[:10]

[1.0,
 0.9914950851356026,
 0.9427895332767988,
 0.9158938129399173,
 0.9058910824116893,
 0.9055519665419229,
 0.9019616028763023,
 0.8844796415003492,
 0.884093236280787,
 0.8829969889355953]

### Threshold Experimentation

In [18]:
for t in [0.05, 0.1, 0.2]:
    print(f"Threshold: {t}")
    print(recommend_anime("Naruto", threshold=t))
    print("-" * 40)

Threshold: 0.05
                                                name  \
0                                 Naruto: Shippuuden   
1                                      Dragon Ball Z   
2                                        Dragon Ball   
3        Naruto: Shippuuden Movie 4 - The Lost Tower   
4  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
5                           Boruto: Naruto the Movie   
6                                        Naruto x UT   
7  Naruto Soyokazeden Movie: Naruto to Mashin to ...   
8                                    Dragon Ball Kai   
9  Boruto: Naruto the Movie - Naruto ga Hokage ni...   

                                               genre  rating     type  \
0  Action, Comedy, Martial Arts, Shounen, Super P...    7.94       TV   
1  Action, Adventure, Comedy, Fantasy, Martial Ar...    8.32       TV   
2  Adventure, Comedy, Fantasy, Martial Arts, Shou...    8.16       TV   
3  Action, Comedy, Martial Arts, Shounen, Super P...    7.53    Movie   
4 

* For similarity thresholds of 0.05, 0.1, and 0.2, the recommended anime list remains
unchanged. This occurs because all top recommended anime have cosine similarity scores
greater than 0.2. Hence, these threshold values are not restrictive enough to filter
recommendations, demonstrating strong similarity clustering in the feature space.

### Analyze Similarity Distribution (Performance Insight)

In [19]:
# Remove self-similarity (diagonal = 1)
sim_values = cosine_sim[np.triu_indices_from(cosine_sim)]

print("Min similarity:", sim_values.min())
print("Max similarity:", sim_values.max())
print("Mean similarity:", sim_values.mean())
print("Median similarity:", np.median(sim_values))

Min similarity: 0.0
Max similarity: 1.0000000000000004
Mean similarity: 0.3255814137683688
Median similarity: 0.2771064630150403


* The cosine similarity distribution shows a minimum of 0.0 and a maximum close to 1.0,
indicating clear separation between unrelated and highly similar anime. The mean similarity
(0.33) and median similarity (0.28) suggest that most anime pairs have low similarity,
with a smaller number of strongly similar pairs. This confirms that the feature
representation and cosine similarity computation are well balanced and effective.

### Genre Overlap Evaluation (Quality Check)

In [20]:
def genre_overlap_score(idx, rec_indices):
    base_genres = set(df.loc[idx, 'genre_list'])
    scores = []

    for i in rec_indices:
        rec_genres = set(df.loc[i, 'genre_list'])
        overlap = len(base_genres & rec_genres) / len(base_genres)
        scores.append(overlap)

    return np.mean(scores)

In [21]:
# Example evaluation
anime_idx = df[df['name'] == "Naruto"].index[0]

similar_indices = np.argsort(cosine_sim[anime_idx])[::-1][1:11]
overlap_score = genre_overlap_score(anime_idx, similar_indices)
print("Average genre overlap score:", overlap_score)

Average genre overlap score: 0.9800000000000001


* The average genre overlap score of 0.98 indicates that the recommended anime share almost
identical genre compositions with the target anime. This demonstrates strong content-based
similarity captured through TF-IDF vectorization and cosine similarity. However, such a
high overlap also suggests reduced recommendation diversity, highlighting a trade-off
between relevance and variety.

### Popularity Bias Check

In [22]:
avg_members_all = df['members'].mean()
avg_members_recs = df.loc[similar_indices, 'members'].mean()
print("Average members (dataset):", avg_members_all)
print("Average members (recommended):", avg_members_recs)

Average members (dataset): 18071.33886448674
Average members (recommended): 165041.3


* The average number of community members for recommended anime (165,041) is significantly
higher than the dataset average (18,071). This indicates that the recommendation system
tends to suggest more popular anime, reflecting effective use of engagement-based features
such as member count.

## Interview Questions

### 1. Difference between User-Based and Item-Based Collaborative Filtering

**User-Based Collaborative Filtering** recommends items by identifying users with similar
preferences. If two users have rated or interacted with items in a similar way, items
liked by one user are recommended to the other. This approach can become less efficient
as the number of users increases.

**Item-Based Collaborative Filtering** recommends items by identifying items that are
similar to each other based on user interaction patterns. If a user likes a particular
item, similar items are recommended. This approach is more scalable and stable, as item
similarities change less frequently than user preferences.

Although collaborative filtering methods are discussed here, the recommendation system
implemented in this assignment is **content-based**, as it relies on anime features rather
than user interaction data.

---

### 2. What is Collaborative Filtering and How Does It Work?

Collaborative filtering is a recommendation technique that makes predictions based on
patterns in user behavior, such as ratings, views, or interactions. It assumes that users
who have shown similar preferences in the past will continue to have similar tastes in
the future.

Collaborative filtering works by analyzing historical user–item interaction data to find
similarities between users or items. Based on these similarities, recommendations are
generated using user-based, item-based, or model-based approaches such as matrix
factorization.